# TP Jour 4 — Inférence Statistique, Segmentation Clients RFM & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement nos intuitions métier avec des **tests statistiques non-paramétriques** (`scipy.stats` : **Mann-Whitney U**, **Kruskal-Wallis**, corrélation de **Spearman**).
2. Construire une **segmentation clients RFM** (*Recency, Frequency, Monetary*) hybride : agrégation distribuée sur **PySpark**, puis clustering **K-Means** standardisé et écriture dans `/lakehouse/gold/customer_rfm`.
3. Connecter l'ensemble au dashboard **Streamlit** (`app/dashboard.py`) et préparer votre livrable final.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_RFM_Stats_Dashboard")
    .getOrCreate()
)

## 1. Tests Statistiques d'Hypothèses sur les Délais et la Satisfaction Client

### Question métier #1 : L'écart de délai de livraison entre le Sud-Est (`SP`, `RJ`, `MG`) et le Nord-Est (`BA`, `PE`, `CE`) est-il statistiquement significatif ?
- **Pourquoi ne pas utiliser un test *t* de Student classique ?** Comme démontré au Jour 3, `delivery_days` présente une forte asymétrie positive (*skewness > 1*) et des valeurs extrêmes qui violent l'hypothèse de normalité.
- **Choix méthodologique :** Le test non-paramétrique de **Mann-Whitney U** compare la distribution des rangs (médianes/dominance stochastique) sans hypothèse gaussienne.

In [ ]:
# Extraction côté Spark uniquement des colonnes utiles (projection pushdown sur Delta Lake)
df_stats = (
    spark.read.format("delta").load("/lakehouse/silver/orders")
    .filter((F.col("order_status") == "delivered") & F.col("delivery_days").isNotNull())
    .select("customer_state", "delivery_days", "delay_days", "review_score")
    .toPandas()
)

sudeste = df_stats.loc[df_stats["customer_state"].isin(["SP", "RJ", "MG"]), "delivery_days"].dropna()
nordeste = df_stats.loc[df_stats["customer_state"].isin(["BA", "PE", "CE"]), "delivery_days"].dropna()

u_stat, p_val_mw = stats.mannwhitneyu(sudeste, nordeste, alternative="two-sided")

print("=== Test de Mann-Whitney U (Sud-Est vs Nord-Est) ===")
print(f"Effectifs        : Sud-Est = {len(sudeste):,} | Nord-Est = {len(nordeste):,}")
print(f"Médianes         : Sud-Est = {sudeste.median():.2f} j | Nord-Est = {nordeste.median():.2f} j")
print(f"Statistique U    : {u_stat:,.0f}")
print(f"p-value          : {p_val_mw:.3e} -> {'Différence hautement significative (H0 rejetée)' if p_val_mw < 0.001 else 'Non significatif'}")

In [ ]:
# Question métier #2 : Quel est l'impact du retard logistique (`delay_days`) sur la note d'avis (`review_score`) ?
valid_rev = df_stats.dropna(subset=["delay_days", "review_score"])
rho, p_val_sp = stats.spearmanr(valid_rev["delay_days"], valid_rev["review_score"])

on_time_scores = valid_rev.loc[valid_rev["delay_days"] <= 0, "review_score"]
late_scores = valid_rev.loc[valid_rev["delay_days"] > 0, "review_score"]

print("=== Impact Logistique sur la Satisfaction Client ===")
print(f"Corrélation de rangs de Spearman (delay_days vs review_score) : rho = {rho:.4f} (p = {p_val_sp:.3e})")
print(f"Note moyenne à l'heure : {on_time_scores.mean():.2f}/5 (médiane = {on_time_scores.median():.1f})")
print(f"Note moyenne en retard : {late_scores.mean():.2f}/5 (médiane = {late_scores.median():.1f})")

## 2. Segmentation Clients RFM (PySpark + K-Means -> `/lakehouse/gold/customer_rfm`)

Attention à une subtilité essentielle du schéma Olist :
- `customer_id` est généré **par commande**.
- **`customer_unique_id`** identifie réellement **le client physique** à travers le temps ! Toute analyse de fidélité ou de fréquence doit impérativement grouper sur `customer_unique_id`.

### Étapes du pipeline :
1. **Agrégation distribuée PySpark** : calcul de `recency_days` (par rapport à la date max du dataset), `frequency` (nombre de commandes distinctes) et `monetary` (chiffre d'affaires cumulé).
2. **Transformation logarithmique (`np.log1p`) + `StandardScaler`** : indispensable avant un K-Means car `monetary` et `recency_days` ont des échelles et asymétries très différentes.
3. **Écriture de la table Gold `/lakehouse/gold/customer_rfm`** immédiatement détectée par `app/dashboard.py`.

In [ ]:
max_date_row = (
    spark.read.format("delta").load("/lakehouse/silver/orders")
    .filter(F.col("order_status") == "delivered")
    .agg(F.max("order_purchase_timestamp").alias("max_ts"))
    .collect()[0]
)
ref_ts = max_date_row["max_ts"]
print(f"Date de référence pour le calcul de Récence : {ref_ts}")

rfm_spark = (
    spark.read.format("delta").load("/lakehouse/silver/orders")
    .filter((F.col("order_status") == "delivered") & F.col("customer_unique_id").isNotNull())
    .groupBy("customer_unique_id")
    .agg(
        F.datediff(F.lit(ref_ts), F.max("order_purchase_timestamp")).alias("recency_days"),
        F.countDistinct("order_id").alias("frequency"),
        F.round(F.sum("total_amount"), 2).alias("monetary"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
        F.first("customer_state", ignorenulls=True).alias("customer_state"),
    )
    .filter(F.col("monetary") > 0)
)

rfm_pd = rfm_spark.toPandas()
print(f"Clients uniques agrégés : {len(rfm_pd):,}")

In [ ]:
# Prétraitement statistique : log1p pour réduire l'asymétrie + centrage-réduction avant K-Means (k=4)
features_log = np.column_stack([
    np.log1p(rfm_pd["recency_days"].clip(lower=0)),
    np.log1p(rfm_pd["frequency"]),
    np.log1p(rfm_pd["monetary"]),
])

X_scaled = StandardScaler().fit_transform(features_log)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_pd["cluster_id"] = kmeans.fit_predict(X_scaled)

# Labellisation métier automatique et déterministe des 4 clusters selon leur profil médian
cluster_profile = rfm_pd.groupby("cluster_id").agg(
    rec_med=("recency_days", "median"),
    freq_mean=("frequency", "mean"),
    mon_med=("monetary", "median"),
)

# Identification du cluster 'Fidèles / Récurrents' (fréquence la plus haute si > 1.2), puis tri par valeur/récence
loyal_cluster = cluster_profile["freq_mean"].idxmax() if cluster_profile["freq_mean"].max() > 1.2 else None
remaining = [c for c in cluster_profile.index if c != loyal_cluster]

# Parmi les restants : celui à plus forte valeur monétaire -> 'Grands Paniers', sinon selon récence
high_val_cluster = cluster_profile.loc[remaining, "mon_med"].idxmax()
others = [c for c in remaining if c != high_val_cluster]
recent_cluster = cluster_profile.loc[others, "rec_med"].idxmin()

label_map = {}
if loyal_cluster is not None:
    label_map[loyal_cluster] = "Fidèles (Multi-achats)"
label_map[high_val_cluster] = "VIP / Grands Paniers"
label_map[recent_cluster] = "Nouveaux / Récents"
for c in cluster_profile.index:
    if c not in label_map:
        label_map[c] = "En sommeil / À réactiver"

rfm_pd["segment"] = rfm_pd["cluster_id"].map(label_map)

print("=== Profil statistique des segments RFM ===")
print(
    rfm_pd.groupby("segment")
    .agg(
        nb_clients=("customer_unique_id", "count"),
        recence_mediane=("recency_days", "median"),
        freq_moyenne=("frequency", "mean"),
        panier_median=("monetary", "median"),
    )
    .sort_values("nb_clients", ascending=False)
    .to_string()
)

In [ ]:
# Écriture de la table finale `/lakehouse/gold/customer_rfm` sur HDFS (Delta Lake)
df_gold_rfm = spark.createDataFrame(rfm_pd)

(
    df_gold_rfm
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("/lakehouse/gold/customer_rfm")
)

print("Table Gold /lakehouse/gold/customer_rfm enregistrée sur HDFS !")

## 3. Validation Finale de l'Architecture Médaillon & Dashboard Streamlit

Vérifions l'inventaire complet de notre couche **Gold** sur HDFS, puis rendez-vous sur **http://localhost:8501** (bouton **Actualiser le cache**) pour explorer le tableau de bord complet !

In [ ]:
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
gold_dir = jvm.org.apache.hadoop.fs.Path("/lakehouse/gold")

print("=== Tables Delta disponibles dans /lakehouse/gold/ ===")
for status in fs.listStatus(gold_dir):
    name = status.getPath().getName()
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{name}").count()
    print(f" - /lakehouse/gold/{name:<18} : {cnt:>8,} lignes")

### Pour aller plus loin dans votre projet (`app/dashboard.py`)
1. Ajoutez dans `app/dashboard.py` un filtre interactif par État (`st.multiselect` sur `kpi_states`) et affichez le rapport `freight_share_pct` vs `late_rate_pct`.
2. Pour chaque graphique ajouté au dashboard, intégrez un bloc `st.expander("Requête SQL & Justification statistique")` exposant la requête Spark SQL exécutée en amont et justifiant le choix de vos indicateurs (médiane, IQR, P90, test de Mann-Whitney, transformation log1p).